# Agent Evaluation

In [ ]:
import os
import openai
from openai import AzureOpenAI
import csv
import time
import re
from typing import Dict, List, Any
import pandas as pd
import sys
import sqlite3
from pathlib import Path
from dotenv import load_dotenv
import json
from datetime import datetime
import re
import time

# Add project root to path
# Note: Update this path according to your local setup
# project_root = "/path/to/your/project"
# src_path = os.path.join(project_root, "src")
# if os.path.exists(src_path):
#     sys.path.append(src_path)

# Import original text2sql generators
from text2sql.engine.generation import AzureGenerator, GCPGenerator
from text2sql.engine.generation.postprocessing import extract_first_code_block

# Load environment variables
load_dotenv()


True

In [ ]:
# Initialize original text2sql generators
def initialize_generator():
    """Initialize the GCP generator using original text2sql"""
    api_key = os.environ.get("GCP_KEY")
    if not api_key:
        raise ValueError("GCP_KEY not found in environment variables")
    
    generator = GCPGenerator(
        api_key=api_key,
        model="gemini-2.5-flash-lite",
    )
    return generator

# Initialize the generator
generator = initialize_generator()

In [101]:
# def parse_score_and_reasoning(content: str):
#     block = extract_first_code_block(content) or content

#     # json first
#     for cand in (block, content):
#         try:
#             data = json.loads(cand)
#         except Exception:
#             try:
#                 data = json.loads(cand.replace("'", '"')) 
#             except Exception:
#                 data = None
#         if isinstance(data, dict):
#             score = data.get("Score", data.get("score"))
#             reason = data.get("Reasoning", data.get("reason", ""))
#             try:
#                 score = float(score)
#             except Exception:
#                 score = None
#             if score is not None:
#                 return max(0, min(5, score)), (str(reason).strip() or content)

#     # not json
#     score_re = re.search(r'["\']?\bscore\b["\']?\s*[:=]\s*("?)([0-5](?:\.\d+)?)\1', content, re.I|re.S)
#     reason_re = re.search(r'["\']?\breason(?:ing)?\b["\']?\s*[:=]\s*(")(.*?)\1', content, re.I|re.S)
#     if score_re:
#         score = float(score_re.group(1))
#         reason = reason_re.group(1).strip() if reason_re else content
#         return max(0, min(5, score)), reason

#     # pure text
#     score_re = re.search(r'\bScore\b\s*[:=]\s*([0-5](?:\.\d+)?)', content, re.I)
#     reason_re = re.search(r'\bReason(?:ing)?\b\s*[:=]\s*(.*)', content, re.I|re.S)
#     if score_re:
#         score = float(score_re.group(1))
#         reason = reason_re.group(1).strip() if reason_re else content
#         return max(0, min(5, score)), reason

#     return 0, content

In [ ]:
# categories
METRIC_CATEGORIES = {
    "Structure": [
        "Argument Soundness",
        "Logical Coherence",
        "Verbosity",
    ],
    "Factuality": [
        "Data Retrieval Accuracy",
        "External Information Accuracy",
    ],
    "Data Sense": [
        "Information Adequacy",
        "Trend Awareness",
        "Model Selection rationale", # only for type3(numerical)
    ],
    "Insightfulness": [
        "Out-of-the-box Thinking",
        "Root Cause Depth", # only for type2
        "Assumption Appropriateness", # only for type3
    ],
    "Operational Implementability": [  # only for type4
        "Actionability",
        "Time-Based Planning",
    ],
    "Purpose Alignment": [ # only for type4
        "Goal Orientation",
        "Stakeholder Orientation",
    ],
    "Compliance": [ # only for type4
        "Risk Management",
        "Regulatory Complaince",
        "Ethical Responsibility",
    ],
}

# definition of metrics
METRIC_DEFINITIONS = {
    # Structure
    "Argument Soundness":
        "To assess whether the subpoints of answer are Mutually Exclusive and Collectively Exhaustive (MECE), a core part of the Minto Pyramid Principle.",
    "Logical Coherence":
        "(1)Contextual alignment: To assess whether the answer aligns with the context of the question (e.g., timeframe, entities). (2)Reasoning alignment: To assess whether the chain of reasoning is complete and sound.",
    "Verbosity":
        "To assess whether the answer is concise and not overly detailed.",
    # Factuality
    "Data Retrieval Accuracy":
        "To assess whether the data included in the answer can be correctly retrieved from the database.",
    "External Information Accuracy":
        "To assess whether the reference link of external information exists and corresponds to the external information.",
    # Data Sense
    "Information Adequacy":
        "To assess whether internal or/and external information included in the answer is adequate to answer the question.",
    "Trend Awareness":
        "To assess whether the external and internal trends are explored in the answer.",
    "Model Selection rationale":
        "To assess whether an rational model is chosen to make predictions.",
    # Insightfulness
    "Out-of-the-box Thinking": 
        "To assess if the analysis introduces conceptual advancements or solutions outside common analytical parameters.",
    "Root Cause Depth":
        "To assess whether the analysis moves beyond surface-level symptoms to identify plausible, well-supported root causes.",
    "Assumption Appropriateness":
        "To assess (1)whether the assumption is appropriate and realistic, (2)whether the results match the assumptions.",
    # Operational Implementability
    "Actionability":
        "To assess whether the recommendation is executable and concrete.",
    "Time-Based Planning":
        "To assess whether the recommendation includes specific implementation timeline.",
    # Purpose Alignment
    "Goal Orientation":
        "To assess whether the recommendation is associated with reachable and motivational goals.",
    "Stakeholder Orientation":
        "To assess if the recommendation explicitly considers and balances the interests and impacts on key non-shareholder stakeholders (e.g., employees, customers, community).",
    # Compliance    
    "Risk Management":  
        "To assess the inclusion of adequate risk warnings, identifications of potential risk points, and provision of risk mitigation suggestions.",
    "Regulatory Complaince":
        "To assess adherence to laws, regulations, and industry standards.",
    "Ethical Responsibility":
        "To assess if the analysis and recommendations uphold principles of AI ethics (e.g., fairness, well-being, non-discrimination), moving beyond mere legal adherence.",
}


all_metrics = [m for cat in METRIC_CATEGORIES.values() for m in cat]
metric_definitions = {m: METRIC_DEFINITIONS[m] for m in all_metrics}

metric_to_category = {
    m: cat
    for cat, metrics in METRIC_CATEGORIES.items()
    for m in metrics
}

# weights
# CATEGORY_WEIGHTS = {
#     "Structure": 0.35,
#     "Factuality": 0.35,
#     # ...
# }

# METRIC_WEIGHTS = {
#     "Argument Soundness": 0.12,
#     "Logical Coherence": 0.12,
#     "Verbosity": 0.11,
#     "Data Retrieval Accuracy": 0.20,
#     "External Information Accuracy": 0.22,
#     "Information Adequacy": 0.23,
# }


In [ ]:
def get_eval_criteria_argument_soundness():
    eval_criteria_argument_soundness_text = """
**Argument Soundness Evaluation Criteria:**
Score: 0
Criteria: The response is disorganized, lacks a main idea, and has redundant or illogical subpoints, making it not MECE.
Example: Our sales are down. We need to focus on marketing and then also on our customer base, especially the new ones. Also, products.

Score: 3
Criteria: The response has a central idea but is only partially MECE; subpoints may overlap or not be fully exhaustive. The logic requires some effort to follow.
Example: Sales are down because of marketing and sales team performance. We should also look at our promotional strategies.

Score: 5
Criteria: The response is a perfectly coherent pyramid, with a single main point and subpoints that are perfectly MECE and easy to grasp. 
Example: Sales are down due to Price and Volume. We will analyze each of these areas separately to find the root cause.
"""
    return eval_criteria_argument_soundness_text

def get_eval_criteria_logical_coherence():
    eval_criteria_logical_coherence_text = """
**Logical Coherence Evaluation Criteria:**
Score: 0
Criteria: The response contradicts the context or is internally contradictory, with fundamentally flawed or incomplete reasoning.
Example: Our Q3 sales were $100M, which is a 10% decrease from last quarter's $80M.

Score: 3
Criteria: The response is mostly consistent but has minor logical gaps, making it difficult to follow. 
Example: Our Q3 sales grew significantly because we launched a new product, which increased our market share.

Score: 5
Criteria: The response is perfectly aligned with the context and has a complete, sound, and seamless chain of reasoning.
Example: Our Q3 sales grew by 10% from Q2, from $80M to $88M, which was a direct result of our new marketing campaign targeting urban areas.
"""
    return eval_criteria_logical_coherence_text

def get_eval_criteria_verbosity():
    eval_criteria_verbosity_text = """
**Verbosity Evaluation Criteria:**
Score: 0
Criteria: The response is overly verbose, repetitive, and uninformative.
Example: In order to improve our sales performance, it is imperative that we strategically and proactively implement a new initiative designed to enhance overall revenue generation and expand our presence in the market.

Score: 3
Criteria: The response is of reasonable length but could be more concise, containing some redundant details.
Example: To improve our sales, we should put a new initiative in place to increase revenue and market presence.

Score: 5
Criteria: The response is concise and to the point, using the minimum words to convey maximum information efficiently. 
Example: To increase sales, we will launch a new initiative to expand our market presence.
"""
    return eval_criteria_verbosity_text

def get_eval_criteria_data_retrieval_accuracy():
    eval_criteria_data_retrieval_accuracy_text = """
**Data Retrieval Accuracy Evaluation Criteria:**
Score: 0
Criteria: The response contains incorrect or un-retrievable data, a severe hallucination.
Example: The Q3 financial report shows an increase in revenue of 50%, with total sales of $200M. (Database shows revenue decreased and sales were $100M)

Score: 1
Criteria: The response's data is 100% accurate and verifiable against the source, with no fabrication. 
Example: The Q3 financial report shows sales of exactly $104.8M. (Database shows sales were $104.8M)
"""
    return eval_criteria_data_retrieval_accuracy_text

def get_eval_criteria_external_information_accuracy():
    eval_criteria_external_information_accuracy_text = """
**External Information Accuracy Evaluation Criteria:**
Score: 0
Criteria: The response contains fabricated information or uses non-existent/unsupported references, a critical hallucination. No external information is mentioned or all mentioned external information is clearly false.
Example: A recent study found that 90% of consumers prefer eco-friendly products. (No source mentioned, or link leads to a non-existent study)

Score: 1
Criteria: The response mentions external information but provides no references, sources, or citations at all. The information may be plausible but is completely unverifiable.
Example: Industry best practices suggest that free-shipping thresholds typically increase AOV by 10-15%. (No source or reference provided)

Score: 2
Criteria: The response mentions external information with vague or incomplete references (e.g., only source names without links, or generic mentions like "industry data" or "research shows"). The information cannot be verified.
Example: According to Google Search/Wordstream data, Q4 ROAS typically increases. (Source name mentioned but no specific link or verifiable reference)

Score: 3
Criteria: The response provides some external information with partial references (e.g., source names and general descriptions), but lacks direct links or specific citations. Some information may be verifiable through general knowledge.
Example: A study from XYZ Research shows that 90% of consumers prefer eco-friendly products. The study was published in 2023. (Source and publication info provided but no direct link)

Score: 4
Criteria: The response provides external information with mostly complete references, including source names and some verifiable details. Most external claims have corresponding references, though some links may be missing or not directly accessible.
Example: According to a 2023 report from XYZ Research (available at xyzresearch.com/reports/2023), 90% of consumers prefer eco-friendly products. (Link provided but may require additional navigation to verify)

Score: 5
Criteria: All external information is factually correct and fully supported by provided references with direct, verifiable links. Every external claim has a corresponding, accessible reference that can be verified. The output is completely trustworthy and verifiable.
Example: A recent study from XYZ Research (https://xyzresearch.com/studies/2023/consumer-preferences, published March 2023) shows that 90% of consumers prefer eco-friendly products. The study surveyed 10,000 participants across 50 countries. (Direct link provided and specific details given)
"""
    return eval_criteria_external_information_accuracy_text

def get_eval_criteria_information_adequacy():
    eval_criteria_information_adequacy_text = """
**Information Adequacy Evaluation Criteria:**
Score: 0
Criteria: The response is inadequate, missing critical information or misinterpreting the user's intent.
Example: The Q3 revenue decline was caused by the cancellation of an insignificant user's account.

Score: 3
Criteria: The response is functional but misses critical details or context, leaving significant follow-up questions. 
Example: The Q3 revenue decline was caused by decreased sales in the US market.

Score: 5
Criteria: The response is comprehensive, providing all necessary information to fully address the query and anticipate user needs.
Example: The Q3 revenue decline was due to a 10% drop in unit volume, driven by increased competition and a new competitor's product launch that targeted our key demographic.
"""
    return eval_criteria_information_adequacy_text

def get_eval_criteria_trend_awareness():
    eval_criteria_trend_awareness_text = """
**Trend Awareness Evaluation Criteria:**
Score: 0
Criteria: The response is static, providing data without considering trends.
Example: The sales in next November will be the same as October. (Igoring Black Friday trend)

Score: 3
Criteria: The response acknowledges trends but fails to integrate them meaningfully into the analysis.
Example: Next month is holiday season, but we do not need to take this into account when managing inventories.

Score: 5
Criteria: The response identifies and expertly integrates relevant trends into the analysis, providing crucial context and a forward-looking perspective.
Example: In summer, people tend to go to beaches, so the hotels near beaches might increase prices by 20%.
"""
    return eval_criteria_trend_awareness_text

def get_eval_criteria_model_selection_rationale():
    eval_criteria_model_selection_rationale_text = """
**Model Selection rationale Evaluation Criteria:**
Score: 0
Criteria: An inappropriate or unsuited model is used for the prediction, showing a lack of fundamental data science understanding.
Example: We used a simple linear regression model to predict next year's sales, ignoring seasonality and market shifts.

Score: 3
Criteria: A reasonable model is selected, but the choice is not justified, and its limitations are not acknowledged.
Example: We used an ARIMA model to predict sales.

Score: 5
Criteria: A rational and well-justified model is selected for the prediction, with a clear explanation of its reasoning and limitations.
Example: We chose a Prophet model to forecast sales due to its strong performance on time-series data with clear seasonal and holiday effects, which are present in our dataset.
"""
    return eval_criteria_model_selection_rationale_text

def get_eval_criteria_out_of_the_box_thinking():
    eval_criteria_out_of_the_box_thinking_text = """
**Out-of-the-Box Thinking Evaluation Criteria:**
Score: 0
Criteria: The reasoning is conventional, relying only on standard formulas or predetermined steps.
Example: Used the standard 5 Forces analysis without adjustment.

Score: 3
Criteria: The reasoning adapts a standard framework to the problem context but does not introduce a novel concept.
Example: Used a modified Profitability Tree to incorporate a new variable.

Score: 5
Criteria: The response develops a novel analytical approach or heuristic to solve the problem, going beyond expected logical steps.
Example: Developed a new heuristic to model customer retention based on unstructured social data, resulting in a novel segmentation.
"""
    return eval_criteria_out_of_the_box_thinking_text

def get_eval_criteria_root_cause_depth():
    eval_criteria_root_cause_depth_text = """
**Root Cause Depth Evaluation Criteria:**
Score: 0
Criteria: The response only describes symptoms and offers no root cause analysis.
Example: Sales are down because customers are not buying our products.

Score: 3
Criteria: The analysis is shallow or based on unsupported assumptions, providing weak conclusions.
Example: Sales are down due to a lack of marketing and high prices.

Score: 5
Criteria: The analysis is thorough and multi-faceted, identifying plausible, well-supported root causes and proposing mitigations. 
Example: Sales are down due to a new competitor that entered the market with lower prices and a superior product. This has made our existing product non-competitive, leading to a 20% drop in volume.
"""
    return eval_criteria_root_cause_depth_text

def get_eval_criteria_assumption_appropriateness():
    eval_criteria_assumption_appropriateness_text = """
**Assumption Appropriateness Evaluation Criteria:**
Score: 0
Criteria: The analysis is based on unrealistic or unstated assumptions, invalidating the conclusion.
Example: Our new product will be a success. (Based on no data)

Score: 3
Criteria: Some assumptions are appropriate but are not clearly stated or justified, leading to a confusing analysis.
Example: We assume that Gen Z will respond positively to our new product due to its viral potential.

Score: 5
Criteria: All assumptions are clearly stated, justified, and realistic, with conclusions that logically follow and are fully consistent.
Example: We assume a 5% increase in market share in Q4, based on our successful pilot program and customer feedback showing a high intent to purchase.
"""
    return eval_criteria_assumption_appropriateness_text

def get_eval_criteria_actionability():
    eval_criteria_actionability_text = """
**Actionability Evaluation Criteria:**
Score: 0
Criteria: The recommendation is too vague or abstract to be executed, with no concrete steps.
Example: The company should improve its online presence.

Score: 3
Criteria: The recommendation is high-level and lacks specific, measurable steps for implementation.
Example: The company should launch a social media campaign and update its website.

Score: 5
Criteria: The recommendation is highly actionable, concrete, and includes specific steps tied to business KPIs.
Example: Launch a TikTok campaign with five influencer partners, with a budget of $50,000, and redesign the homepage with A/B testing on two new layouts.
"""
    return eval_criteria_actionability_text

def get_eval_criteria_time_based_planning():
    eval_criteria_time_based_planning_text = """
**Time-Based Planning Evaluation Criteria:**
Score: 0
Criteria: The recommendation lacks any timeline or sense of urgency.
Example: We recommend a new marketing campaign.

Score: 3
Criteria: The recommendation includes a vague, general timeline that does not allow for effective planning.
Example: We will launch the new campaign in the near future.

Score: 5
Criteria: The recommendation has a specific, realistic, and well-justified timeline with clear milestones for effective planning.
Example: We will launch the new marketing campaign in Q4, with a goal of completion by December 15th.
"""
    return eval_criteria_time_based_planning_text

def get_eval_criteria_goal_orientation():
    eval_criteria_goal_orientation_text = """
**Goal Orientation Evaluation Criteria:**
Score: 0
Criteria: The recommendation is not linked to any clear goals or business value.
Example: We suggest you hire a new VP of Marketing.

Score: 3
Criteria: The recommendation is linked to a general goal that is not specific or measurable.
Example: We suggest you hire a new VP of Marketing to increase brand awareness.

Score: 5
Criteria: The recommendation is directly tied to a specific, measurable, and achievable (SMART) business goal with a clear value proposition.
Example: We suggest you hire a new VP of Marketing with the goal of increasing our market share by 5% and our social media engagement by 20% in the next fiscal year.
"""
    return eval_criteria_goal_orientation_text

def get_eval_criteria_stakeholder_orientation():
    eval_criteria_stakeholder_orientation_text = """
**Stakeholder Orientation Evaluation Criteria:**
Score: 0
Criteria: The recommendation focuses exclusively on financial metrics (e.g., ROI) and ignores all non-shareholder impacts.
Example: Recommend mass layoffs to cut costs by 20% without considering employee morale or retention.

Score: 3
Criteria: Mentions key stakeholders (e.g., customers) but does not integrate their perspectives into the final recommendation or mitigation plan.
Example: The plan may impact employees, but the financial return justifies it.

Score: 5
Criteria: The recommendation explicitly balances financial gains with stakeholder well-being, including measures to mitigate negative effects on non-shareholder groups.
Example: Recommend a phased automation rollout, offsetting job losses with a robust retraining program for high-value roles.
"""
    return eval_criteria_stakeholder_orientation_text

def get_eval_criteria_risk_management():
    eval_criteria_risk_management_text = """
**Risk Management Evaluation Criteria:**
Score: 0
Criteria: The response fails to identify any risks or contains content that exacerbates risk.
Example: The new product launch will be a guaranteed success.

Score: 3
Criteria: Some risks are identified, but warnings or mitigation suggestions are inadequate.
Example: A new product launch could fail, so we should be careful.

Score: 5
Criteria: The response provides comprehensive risk warnings, identifies potential risk points, and offers concrete mitigation suggestions.
Example: We recognize the risk of market saturation and mitigate this by conducting a thorough competitive analysis, allocating a contingency budget for unexpected costs, and a phased rollout to a test market.
"""
    return eval_criteria_risk_management_text

def get_eval_criteria_regulatory_compliance():
    eval_criteria_regulatory_compliance_text = """
**Regulatory Compliance Evaluation Criteria:**
Score: 0
Criteria: The response is non-compliant with relevant laws and regulations, posing a severe risk.
Example: To bypass data privacy regulations, you should use customer data from our partners without their explicit consent.

Score: 3
Criteria: The response is generally compliant but lacks specific disclaimers or fails to fully address all regulatory requirements.
Example: Our new data collection process will adhere to all regulations, but we will not be adding a legal disclaimer to our website.

Score: 5
Criteria: The response is fully compliant with all applicable laws and regulations, including adequate disclaimers and privacy protocols.
Example: All of our new data collection processes are fully compliant with GDPR and CCPA, and we have implemented a clear, user-facing privacy policy and consent form to ensure full legal adherence.
"""
    return eval_criteria_regulatory_compliance_text

def get_eval_criteria_ethical_responsibility():
    eval_criteria_ethical_responsibility_text = """
**Ethical Responsibility Evaluation Criteria:**
Score: 0
Criteria: The recommendation is demonstrably biased or perpetuates known social inequalities, or ignores explicit ethical consequences.
Example: The loan risk model shows bias against applicants from specific zip codes; recommend proceeding without mitigation.

Score: 3
Criteria: Includes a general statement about "fairness" but lacks concrete, measurable steps to mitigate identified bias or ensure equitable outcomes.
Example: We should try to be fair, but the model output is final.

Score: 5
Criteria: The recommendation includes specific, actionable steps (e.g., bias monitoring, disparate impact analysis) to ensure equitable outcomes and promote the well-being of the affected demographic.
Example: Implement a specific re-weighting algorithm to address observed demographic bias in the output and monitor fairness metrics in real-time.
"""
    return eval_criteria_ethical_responsibility_text


# Score: 0
# Criteria: 
# Example: 

# Score: 3
# Criteria: 
# Example: 

# Score: 5
# Criteria: 
# Example: 

# Multi-Agent Coordination Protocol Documentation

## Coordination Architecture

The multi-agent evaluation system follows a **hierarchical coordination protocol** with three main phases:

### Phase 1: Metric Selection (Discriminator Agent)
- **Input**: `(question_type, question, answer)`
- **Output**: `Dict[Category, List[Metric]]` - Selected metrics per category
- **Process**: 
  - For type2: Returns fixed metric set
  - For type3/type4: Uses LLM to dynamically determine additional metrics
  - Makes binary decisions (YES/NO) for optional metrics

### Phase 2: Parallel Scoring (Category Scoring Agents)
- **Input**: `(question, answer, metric)` for each selected metric
- **Output**: `{"metric": str, "score": int, "reasoning": str}`
- **Process**: 
  - Each category agent evaluates metrics in its domain
  - LLM generates score (0-5) and reasoning
  - JSON parsing with fallback regex extraction

### Phase 3: Result Aggregation (Comprehensive Evaluator)
- **Input**: All scoring results from Phase 2
- **Output**: Flattened evaluation results with all scores and reasoning
- **Process**: 
  - Collects results from all category agents
  - Flattens nested structure for CSV export
  - Adds metadata (question_number, question_type)

## Communication Protocol

```
[ComprehensiveEvaluator]
    │
    ├─→ [DiscriminatorAgent.determine_metrics()]
    │   └─→ Returns: Dict[Category, List[Metric]]
    │
    └─→ [For each Category in selected_metrics]
        └─→ [CategoryScoringAgent.evaluate_metric()]
            └─→ Returns: {"metric": str, "score": int, "reasoning": str}
```

## Tool Use Details

### LLM Interaction Protocol
- **Model**: Gemini 2.5 Flash Lite (via GCPGenerator)
- **Temperature**: Default (not explicitly set)
- **Prompt Engineering**: 
  - System prompt for scoring agents
  - Structured JSON output format
  - Fallback regex parsing for non-JSON responses

### Error Handling
- **API Failures**: Returns score=0 with error message in reasoning
- **Parsing Failures**: Falls back to regex extraction, then defaults to 0
- **Missing Metrics**: Returns score=0 with "No evaluation criteria available"


In [ ]:
# discriminator agent
class DiscriminatorAgent:
    """
    Discriminator Agent: Responsible for metric selection based on question type and content.
    
    Coordination Role:
    - Acts as the first phase in the multi-agent evaluation pipeline
    - Determines which metrics should be evaluated for each question-answer pair
    - Uses LLM to make dynamic decisions for optional metrics (type3/type4)
    
    Communication Protocol:
    - Input: (question_type, question, answer)
    - Output: Dict[str, List[str]] - {category: [list of metrics]}
    - Called by: ComprehensiveEvaluator.evaluate_question_answer()
    
    Tool Use:
    - LLM calls for binary classification (YES/NO) for optional metrics
    - No retry mechanism (fails gracefully by defaulting to False)
    """
    def __init__(self, generator):
        """
        Initialize the Discriminator Agent.
        
        Args:
            generator: GCPGenerator instance for LLM interactions
        """
        self.generator = generator
    
    def evaluate_type3_metrics(self, question: str, answer: str) -> Dict[str, List[str]]:
        """
        Evaluate type3 questions with dynamic metric selection based on LLM judgment
        """
        # Base metrics for type3 questions
        # Note: Factuality metrics are not applicable for type3
        # - Data Retrieval Accuracy is temporarily commented out for all types
        # - External Information Accuracy is only applicable to type4 questions
        base_metrics = {
            "Structure": ["Argument Soundness", "Logical Coherence", "Verbosity"],
            "Factuality": [],  # No factuality metrics for type3
            "Data Sense": ["Information Adequacy", "Trend Awareness"],
            "Insightfulness": ["Out-of-the-box Thinking", "Assumption Appropriateness"]
        }
        
        # LLM prompts for additional metric evaluation
        # Note: Root Cause Depth is not applicable for type3 questions
        model_selection_prompt = f"""
        Analyze the following question and answer to determine if the question or answer involves numerical prediction.
        
        Question: {question}
        Answer: {answer}
        
        Does this question or answer involve numerical prediction or modeling that would require model selection rationale? Respond with only "YES" or "NO".
        """
        
        # Function to get LLM response
        def get_llm_judgment(prompt: str) -> bool:
            try:
                messages = [
                    {"role": "user", "content": prompt},
                ]
                
                response = self.generator.generate(messages)
                return response.strip().upper() == "YES"
            except Exception as e:
                print(f"Error getting LLM judgment: {e}")
                return False
        
        # Evaluate additional metrics
        # Note: Root Cause Depth is not applicable for type3, so we only check for Model Selection Rationale
        has_model_selection_rationale = get_llm_judgment(model_selection_prompt)
        
        # Add additional metrics if applicable
        if has_model_selection_rationale:
            base_metrics["Data Sense"].append("Model Selection Rationale")
        
        return base_metrics

    def evaluate_type4_metrics(self, question: str, answer: str) -> Dict[str, List[str]]:
        """
        Evaluate type4 questions with dynamic metric selection based on LLM judgment
        """
        # Base metrics for type4 questions
        # Note: 
        # - Data Retrieval Accuracy is temporarily commented out for all types
        # - External Information Accuracy is only applicable to type4 questions
        # - Root Cause Depth and Assumption Appropriateness are not applicable for type4
        base_metrics = {
            "Structure": ["Argument Soundness", "Logical Coherence", "Verbosity"],
            "Factuality": ["External Information Accuracy"],  # Only External Information Accuracy for type4
            "Data Sense": ["Information Adequacy", "Trend Awareness"],
            "Insightfulness": ["Out-of-the-box Thinking"],  # No Root Cause Depth or Assumption Appropriateness
            "Operational Implementability": ["Actionability", "Time-Based Planning"],
            "Purpose Alignment": ["Goal Orientation", "Stakeholder Orientation"],
            "Compliance": ["Risk Management", "Regulatory Compliance", "Ethical Responsibility"],
        }
        
        # LLM prompts for additional metric evaluation
        # Note: Root Cause Depth and Assumption Appropriateness are not applicable for type4 questions
        model_selection_prompt = f"""
        Analyze the following question and answer to determine if the question or answer involves numerical prediction.
        
        Question: {question}
        Answer: {answer}
        
        Does this question or answer involve numerical prediction or modeling that would require model selection rationale? Respond with only "YES" or "NO".
        """
        
        # Function to get LLM response
        def get_llm_judgment(prompt: str) -> bool:
            try:
                messages = [
                    {"role": "user", "content": prompt},
                ]
                
                response = self.generator.generate(messages)
                return response.strip().upper() == "YES"
            except Exception as e:
                print(f"Error getting LLM judgment: {e}")
                return False
        
        # Evaluate additional metrics
        # Note: Root Cause Depth and Assumption Appropriateness are not applicable for type4, so we only check for Model Selection Rationale
        has_model_selection = get_llm_judgment(model_selection_prompt)
        
        # Add additional metrics if applicable
        if has_model_selection:
            base_metrics["Data Sense"].append("Model Selection rationale")
        
        return base_metrics


    def determine_metrics(self, question_type, question, answer):
        """
        Select evaluation metrics based on question type and answer.
        
        Coordination Protocol:
        This is the main interface method called by ComprehensiveEvaluator.
        It implements the metric selection phase of the coordination protocol.
        
        Args:
            question_type: str - One of ['type2', 'type3', 'type4']
            question: str - The question text
            answer: str - The answer text to be evaluated
            
        Returns:
            Dict[str, List[str]]: Dictionary mapping category names to lists of metric names
            Example: {
                "Structure": ["Argument Soundness", "Logical Coherence", "Verbosity"],
                "Factuality": ["Data Retrieval Accuracy", "External Information Accuracy"],
                ...
            }
            
        Coordination Flow:
        1. For type2: Returns fixed metric set (no LLM call needed)
        2. For type3: Calls evaluate_type3_metrics() which uses LLM for dynamic selection
        3. For type4: Calls evaluate_type4_metrics() which uses LLM for dynamic selection
        
        Tool Use:
        - For type3/type4: Makes 2-3 LLM calls for binary classification
        - Each call uses a simple prompt asking YES/NO question
        - Response is parsed as uppercase "YES" or default False
        """
        if question_type == "type2":
            metrics = {
                "Structure": ["Argument Soundness", "Logical Coherence", "Verbosity"],
                "Factuality": ["Data Retrieval Accuracy", "External Information Accuracy"],
                "Data Sense": ["Information Adequacy", "Trend Awareness"],
                "Insightfulness": ["Out-of-the-box Thinking", "Root Cause Depth"]
            }
            return metrics
        elif question_type == "type3":
            return self.evaluate_type3_metrics(question, answer)
        elif question_type == "type4":
            return self.evaluate_type4_metrics(question, answer)


discriminator = DiscriminatorAgent(generator)

In [ ]:
# Category-specific scoring agents
class CategoryScoringAgent:
    def __init__(self, category_name, metrics, generator):
        self.category_name = category_name
        self.metrics = metrics
        self.generator = generator
        
    def get_evaluation_prompt(self, question, answer, metric):
        """Generate evaluation prompt for a specific metric"""
        # Get the evaluation criteria function
        metric_key = metric.lower().replace(' ', '_').replace('-', '_')
        criteria_func = globals().get(f"get_eval_criteria_{metric_key}")
        
        if not criteria_func:
            return None
            
        criteria = criteria_func()
        
        prompt = f"""
You are an expert evaluator. Please evaluate the following answer based on the given question using the {metric} metric.

**Question:** {question}

**Answer:** {answer}

**Metric Definition:**
{metric}: {METRIC_DEFINITIONS.get(metric, 'No definition available')}

**Evaluation Criteria:**
{criteria}

**Instructions:**
Please provide your evaluation in the following JSON format:
{{
    "Score": <score 0-5>,
    "Reasoning": "<brief explanation of the score>"
}}

Please be precise and provide clear reasoning for your score. The output format should be strictly as above. Do not include any other text other than the JSON format.
"""
        return prompt
    
    def evaluate_metric(self, question, answer, metric):
        """Evaluate a single metric"""
        prompt = self.get_evaluation_prompt(question, answer, metric)
        if not prompt:
            return {"metric": metric, "score": 0, "reasoning": "No evaluation criteria available"}
            
        try:
            messages = [
                {"role": "system", "content": "You are an evaluation expert with 10+ years of experience in data science and AI. Provide a numeric score and a short reasoning based on the evaluation criteria."},
                {"role": "user", "content": prompt}
                ]
                
            content = self.generator.generate(messages)
            # score, reasoning = parse_score_and_reasoning(content)

            # Extract score and reasoning from response
            # Try JSON format first (with quotes)
            score_match = re.search(r'"Score"\s*:\s*(\d+)', content, re.IGNORECASE)
            reason_match = re.search(r'"Reasoning"\s*:\s*"([^"]+)"', content, re.IGNORECASE)
            
            # If JSON format doesn't work, try without quotes
            if not score_match:
                score_match = re.search(r'Score\s*:\s*(\d+)', content, re.IGNORECASE)
            if not reason_match:
                reason_match = re.search(r'Reasoning\s*:\s*(.+)', content, re.IGNORECASE)

            score = int(score_match.group(1)) if score_match else 0
            reasoning = reason_match.group(1).strip() if reason_match else content

            return {"metric": metric, "score": score, "reasoning": reasoning}

        except Exception as e:
            return {"metric": metric, "score": 0, "reasoning": f"API error: {e}"}
    
    def evaluate_all_metrics(self, question, answer):
        """Evaluate all metrics in this category"""
        results = {}
        for metric in self.metrics:
            result = self.evaluate_metric(question, answer, metric)
            results[metric] = result
        return results

# Create scoring agents for each category
def create_scoring_agents():
    """Create all category scoring agents"""
    agents = {}
    
    # Structure agent
    agents["Structure"] = CategoryScoringAgent(
        "Structure", 
        ["Argument Soundness", "Logical Coherence", "Verbosity"], 
        generator,
    )
    
    # Factuality agent
    agents["Factuality"] = CategoryScoringAgent(
        "Factuality", 
        ["Data Retrieval Accuracy", "External Information Accuracy"],
        generator,
    )
    
    # Data Sense agent
    agents["Data Sense"] = CategoryScoringAgent(
        "Data Sense", 
        ["Information Adequacy", "Trend Awareness", "Model Selection rationale"], 
        generator,
    )
    
    # Insightfulness agent
    agents["Insightfulness"] = CategoryScoringAgent(
        "Insightfulness", 
        ["Out-of-the-box Thinking", "Root Cause Depth", "Assumption Appropriateness"], 
        generator,
    )
    
    # Operational Implementability agent
    agents["Operational Implementability"] = CategoryScoringAgent(
        "Operational Implementability", 
        ["Actionability", "Time-Based Planning"], 
        generator,
    )
    
    # Purpose Alignment agent
    agents["Purpose Alignment"] = CategoryScoringAgent(
        "Purpose Alignment", 
        ["Goal Orientation", "Stakeholder Orientation"], 
        generator,
    )
    
    # Compliance agent
    agents["Compliance"] = CategoryScoringAgent(
        "Compliance", 
        ["Risk Management", "Regulatory Compliance", "Ethical Responsibility"], 
        generator,
    )
    
    return agents

# Create all scoring agents
scoring_agents = create_scoring_agents()

# Comprehensive evaluation system
class ComprehensiveEvaluator:
    def __init__(self, discriminator_agent, scoring_agents):
        self.discriminator = discriminator_agent
        self.scoring_agents = scoring_agents
    
    def evaluate_question_answer(self, question, answer, question_type):
        """Evaluate a single question-answer pair"""
        # Step 1: Determine which metrics to evaluate using discriminator
        selected_metrics = self.discriminator.determine_metrics(question_type, question, answer)
        
        print(f"Selected metrics for {question_type}: {selected_metrics}")
        
        # Step 2: Evaluate each selected metric using appropriate scoring agent
        evaluation_results = {}
        
        for category, metrics in selected_metrics.items():
            if category in self.scoring_agents:
                print(f"Evaluating {category} metrics: {metrics}")
                category_results = {}
                
                for metric in metrics:
                    result = self.scoring_agents[category].evaluate_metric(question, answer, metric)
                    category_results[metric] = result
                    print(f"  {metric}: Score {result['score']}")
                
                evaluation_results[category] = category_results
        
        return evaluation_results
    
    def evaluate_batch(self, questions, answers, question_types):
        """Evaluate a batch of question-answer pairs"""
        results = []
        
        for i, (question, answer, q_type) in enumerate(zip(questions, answers, question_types)):
            print(f"\nEvaluating question {i+1}/{len(questions)} (Type: {q_type})")
            print(f"Question: {question[:100]}...")
            
            evaluation = self.evaluate_question_answer(question, answer, q_type)
            
            # Flatten results for easier analysis
            flattened_result = {
                'question_number': i + 1,
                'question': question,
                'answer': answer,
                'question_type': q_type
            }
            
            # Add scores and reasoning for each metric
            for category, metrics in evaluation.items():
                for metric, result in metrics.items():
                    metric_key = metric.replace(' ', '_').replace('-', '_')
                    flattened_result[f'{metric_key}_score'] = result['score']
                    flattened_result[f'{metric_key}_reasoning'] = result['reasoning']
            
            results.append(flattened_result)
            
            # Add delay to avoid rate limiting
            time.sleep(2)
        
        return results

# Create the comprehensive evaluator
comprehensive_evaluator = ComprehensiveEvaluator(discriminator, scoring_agents)

In [ ]:
# Load and evaluate - UNIFIED VERSION FOR ALL TYPES
def load_and_evaluate_shopify_data():
    """Load data from shopify_all_answers_gemini2.5.csv and perform comprehensive evaluation"""
    
    # Load the unified CSV file
    # Note: Update this path according to your local setup
    csv_path = "path/to/shopify_all_answers_gemini2.5.csv"  # Update with your actual path
    df = pd.read_csv(csv_path)
    
    print(f"Loaded {len(df)} questions from {csv_path}")
    print(f"Columns: {list(df.columns)}")
    
    # Extract questions and answers
    questions = df['Question'].tolist()
    answers = df['Answer'].tolist()
    question_numbers = df['Question Number'].tolist()
    
    # Determine question types based on question number
    # 101-199 = type1, 201-299 = type2, 301-399 = type3, 401-499 = type4
    question_types = []
    for q_num in question_numbers:
        if 100 <= q_num < 200:
            question_types.append('type1')
        elif 200 <= q_num < 300:
            question_types.append('type2')
        elif 300 <= q_num < 400:
            question_types.append('type3')
        elif 400 <= q_num < 500:
            question_types.append('type4')
        else:
            question_types.append('unknown')
    
    print(f"\nQuestion type distribution:")
    print(f"  Type 1: {question_types.count('type1')}")
    print(f"  Type 2: {question_types.count('type2')}")
    print(f"  Type 3: {question_types.count('type3')}")
    print(f"  Type 4: {question_types.count('type4')}")
    
    # Perform batch evaluation
    results = comprehensive_evaluator.evaluate_batch(questions, answers, question_types)
    
    # Add question numbers to results
    for i, result in enumerate(results):
        result['question_number'] = question_numbers[i]
    
    # Export results
    output_filename = "shopify_all_evaluation_results_gemini2.5.csv"
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_filename, index=False)
    
    print(f"\n✅ Evaluation complete! Results saved to {output_filename}")
    
    return df_results, results

# Run the evaluation
print("=" * 80)
print("EVALUATING SHOPIFY ANSWERS (GEMINI 2.5)")
print("=" * 80)

df_results, evaluation_results = load_and_evaluate_shopify_data()


EVALUATING ANSWERS
Loaded 30 questions from C:\CORGI\appstore\answer-gpt4o\appstore_type2_answers_gpt4o.csv
Columns: ['Question Number', 'Question', 'Answer']

Evaluating question 1/30 (Type: type2)
Question: Why did download numbers in March 2025 much higher than other months?...
Selected metrics for type2: {'Structure': ['Argument Soundness', 'Logical Coherence', 'Verbosity'], 'Factuality': ['Data Retrieval Accuracy', 'External Information Accuracy'], 'Data Sense': ['Information Adequacy', 'Trend Awareness'], 'Insightfulness': ['Out-of-the-box Thinking', 'Root Cause Depth']}
Evaluating Structure metrics: ['Argument Soundness', 'Logical Coherence', 'Verbosity']
  Argument Soundness: Score 3
  Logical Coherence: Score 5
  Verbosity: Score 3
Evaluating Factuality metrics: ['Data Retrieval Accuracy', 'External Information Accuracy']
  Data Retrieval Accuracy: Score 1
  External Information Accuracy: Score 0
Evaluating Data Sense metrics: ['Information Adequacy', 'Trend Awareness']
  Info

In [ ]:
# Load and evaluate
def load_and_evaluate_data_type3():
    """Load data and perform comprehensive evaluation"""
    
    # !! change every time - Load the CSV file
    # Note: Update this path according to your local setup
    csv_path = "path/to/appstore_type3_answers_gpt4o.csv"  # Update with your actual path
    df = pd.read_csv(csv_path)
    
    print(f"Loaded {len(df)} questions from {csv_path}")
    print(f"Columns: {list(df.columns)}")
    
    # Extract questions and answers
    questions = df['Question'].tolist()
    answers = df['Answer'].tolist()
    question_numbers = df['Question Number'].tolist()
    
    # !! change every time - All questions
    question_types = ['type3'] * len(questions)
    
    # Perform batch evaluation
    results = comprehensive_evaluator.evaluate_batch(questions, answers, question_types)
    
    # Add question numbers to results
    for i, result in enumerate(results):
        result['question_number'] = question_numbers[i]
    
    # !! change every time - Export results
    output_filename = "appstore_type3_evaluation_results_gpt4o_gemini2.5.csv"
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_filename, index=False)
    
    return df_results, results

# Run the evaluation
print("=" * 80)
print("EVALUATING ANSWERS")
print("=" * 80)

df_results, evaluation_results = load_and_evaluate_data_type3()


EVALUATING ANSWERS
Loaded 30 questions from C:\CORGI\appstore\answer-gpt4o\appstore_type3_answers_gpt4o.csv
Columns: ['Question Number', 'Question', 'Answer']

Evaluating question 1/30 (Type: type3)
Question: What will be the total download volume for the next quarter?...
Selected metrics for type3: {'Structure': ['Argument Soundness', 'Logical Coherence', 'Verbosity'], 'Factuality': ['Data Retrieval Accuracy', 'External Information Accuracy'], 'Data Sense': ['Information Adequacy', 'Trend Awareness', 'Model Selection Rationale'], 'Insightfulness': ['Out-of-the-box Thinking', 'Assumption Appropriateness', 'Root Cause Depth']}
Evaluating Structure metrics: ['Argument Soundness', 'Logical Coherence', 'Verbosity']
  Argument Soundness: Score 3
  Logical Coherence: Score 5
  Verbosity: Score 0
Evaluating Factuality metrics: ['Data Retrieval Accuracy', 'External Information Accuracy']
  Data Retrieval Accuracy: Score 5
  External Information Accuracy: Score 5
Evaluating Data Sense metrics: 

In [ ]:
# Load and evaluate
def load_and_evaluate_data_type4():
    """Load data and perform comprehensive evaluation"""
    
    # !! change every time - Load the CSV file
    # Note: Update this path according to your local setup
    csv_path = "path/to/appstore_type4_answers_gpt4o.csv"  # Update with your actual path
    df = pd.read_csv(csv_path)
    
    print(f"Loaded {len(df)} questions from {csv_path}")
    print(f"Columns: {list(df.columns)}")
    
    # Extract questions and answers
    questions = df['Question'].tolist()
    answers = df['Answer'].tolist()
    question_numbers = df['Question Number'].tolist()
    
    # !! change every time - All questions
    question_types = ['type4'] * len(questions)
    
    # Perform batch evaluation
    results = comprehensive_evaluator.evaluate_batch(questions, answers, question_types)
    
    # Add question numbers to results
    for i, result in enumerate(results):
        result['question_number'] = question_numbers[i]
    
    # !! change every time - Export results
    output_filename = "appstore_type4_evaluation_results_gpt4o_gemini2.5.csv"
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_filename, index=False)
    
    return df_results, results

# Run the evaluation
print("=" * 80)
print("EVALUATING ANSWERS")
print("=" * 80)

df_results, evaluation_results = load_and_evaluate_data_type4()


EVALUATING ANSWERS
Loaded 30 questions from C:\CORGI\appstore\answer-gpt4o\appstore_type4_answers_gpt4o.csv
Columns: ['Question Number', 'Question', 'Answer']

Evaluating question 1/30 (Type: type4)
Question: How can we increase download completion rates for paid apps?...
Selected metrics for type4: {'Structure': ['Argument Soundness', 'Logical Coherence', 'Verbosity'], 'Factuality': ['Data Retrieval Accuracy', 'External Information Accuracy'], 'Data Sense': ['Information Adequacy', 'Trend Awareness'], 'Insightfulness': ['Out-of-the-box Thinking', 'Root Cause Depth', 'Assumption Appropriateness'], 'Operational Implementability': ['Actionability', 'Time-Based Planning'], 'Purpose Alignment': ['Goal Orientation', 'Stakeholder Orientation'], 'Compliance': ['Risk Management', 'Regulatory Compliance', 'Ethical Responsibility']}
Evaluating Structure metrics: ['Argument Soundness', 'Logical Coherence', 'Verbosity']
  Argument Soundness: Score 5
  Logical Coherence: Score 0
  Verbosity: Score 

In [ ]:
# Optional: Test with first 3 questions only (uncomment to run)
def test_type2_evaluation():
    """Test evaluation with first 3 questions only"""
    
    # Load the CSV file
    # Note: Update this path according to your local setup
    csv_path = "path/to/appstore_type2_answers_gpt4o.csv"  # Update with your actual path
    df = pd.read_csv(csv_path)
    
    # Take only first 3 questions for testing
    test_df = df.head(5)
    
    print(f"Testing with first {len(test_df)} questions from {csv_path}")
    
    # Extract questions and answers
    questions = test_df['Question'].tolist()
    answers = test_df['Answer'].tolist()
    question_numbers = test_df['Question Number'].tolist()
    
    # All type2 questions
    question_types = ['type2'] * len(questions)
    
    print(f"\nStarting test evaluation of {len(questions)} type3 questions...")
    
    # Perform batch evaluation
    results = comprehensive_evaluator.evaluate_batch(questions, answers, question_types)
    
    # Add question numbers to results
    for i, result in enumerate(results):
        result['question_number'] = question_numbers[i]
    
    # Export results
    output_filename = "appstore_type4_evaluation_results_test.csv"
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_filename, index=False)
    
    
    return df_results, results

# Uncomment the line below to run test evaluation with first 3 questions
test_df_results, test_evaluation_results = test_type2_evaluation()


Testing with first 5 questions from C:\CORGI\appstore\answer-gpt4o\appstore_type2_answers_gpt4o.csv

Starting test evaluation of 5 type3 questions...

Evaluating question 1/5 (Type: type2)
Question: Why did download numbers in March 2025 much higher than other months?...
Selected metrics for type2: {'Structure': ['Argument Soundness', 'Logical Coherence', 'Verbosity'], 'Factuality': ['Data Retrieval Accuracy', 'External Information Accuracy'], 'Data Sense': ['Information Adequacy', 'Trend Awareness'], 'Insightfulness': ['Out-of-the-box Thinking', 'Root Cause Depth']}
Evaluating Structure metrics: ['Argument Soundness', 'Logical Coherence', 'Verbosity']
  Argument Soundness: Score 0
  Logical Coherence: Score 0
  Verbosity: Score 0
Evaluating Factuality metrics: ['Data Retrieval Accuracy', 'External Information Accuracy']
  Data Retrieval Accuracy: Score 0
  External Information Accuracy: Score 0
Evaluating Data Sense metrics: ['Information Adequacy', 'Trend Awareness']
  Information Ad

In [ ]:
# Comprehensive evaluation function for shopify type2
# Evaluate all answer models with all evaluator models

import os
from pathlib import Path

# Create evaluation folder if it doesn't exist
EVALUATION_FOLDER = "evaluation"
os.makedirs(EVALUATION_FOLDER, exist_ok=True)
print(f"✅ Evaluation folder created/verified: {EVALUATION_FOLDER}/")

def evaluate_shopify_type2_all_models():
    """
    Evaluate all shopify type2 answers using multiple evaluator models.
    
    Evaluates:
    - 4 answer models: gpt5, gemini2.5pro, llama4, qwen3
    - 3 evaluator models: gpt5, gemini2.5pro, llama4
    
    Results saved as: shopify_type2_{answer_model}_{evaluator_model}.csv
    """
    dataset = "shopify"
    question_type = "type2"
    
    # Answer models to evaluate (models that generated the answers)
    answer_models = ["gpt5", "gemini2.5pro", "llama4", "qwen3"]
    # Evaluator models (models used for evaluation)
    evaluator_models = ["gpt5", "gemini2.5pro", "llama4"]
    
    print("=" * 80)
    print("SHOPIFY TYPE2 COMPREHENSIVE EVALUATION")
    print("=" * 80)
    print(f"Answer Models: {answer_models}")
    print(f"Evaluator Models: {evaluator_models}")
    print(f"Total evaluations: {len(answer_models)} × {len(evaluator_models)} = {len(answer_models) * len(evaluator_models)}")
    print("=" * 80)
    
    all_evaluation_results = {}
    
    # For each answer model
    for answer_model in answer_models:
        print(f"\n{'='*80}")
        print(f"Processing Answer Model: {answer_model.upper()}")
        print(f"{'='*80}")
        
        # Path to answer file
        answer_file = f"answers/answers_type2/{dataset}_type2_answers/{answer_model}/{answer_model}_type2_answers.csv"
        
        if not os.path.exists(answer_file):
            print(f"⚠️  Answer file not found: {answer_file}")
            continue
        
        # Load answers
        df = pd.read_csv(answer_file)
        print(f"✅ Loaded {len(df)} questions from {answer_file}")
        
        # Check which column contains the answer
        answer_column = None
        if 'Answer' in df.columns:
            answer_column = 'Answer'
        elif 'General Answers' in df.columns:
            answer_column = 'General Answers'
        else:
            print(f"⚠️  Could not find Answer column in {answer_file}")
            print(f"Available columns: {list(df.columns)}")
            continue
        
        all_evaluation_results[answer_model] = {}
        
        # For each evaluator model
        for evaluator_model in evaluator_models:
            print(f"\n{'='*80}")
            print(f"Evaluating with: {evaluator_model.upper()} (Answer Model: {answer_model.upper()})")
            print(f"{'='*80}")
            
            try:
                # Initialize evaluator generator
                eval_generator = initialize_generator(model_key=evaluator_model, use_openrouter=True)
                
                # Create evaluator
                eval_discriminator = DiscriminatorAgent(eval_generator)
                eval_scoring_agents = create_scoring_agents_with_generator(eval_generator)
                eval_evaluator = ComprehensiveEvaluator(eval_discriminator, eval_scoring_agents)
                
                # Store results for this combination
                all_results = []
                
                # Evaluate each question
                for idx, row in df.iterrows():
                    question_number = row.get('Question Number', idx + 1)
                    question = row['Question']
                    answer = row[answer_column]
                    
                    print(f"\n  Question {question_number}/{len(df)}: {question[:80]}...")
                    
                    # Evaluate
                    evaluation_results = eval_evaluator.evaluate_question_answer(
                        question, answer, question_type
                    )
                    
                    # Flatten results
                    flattened_result = {
                        'question_number': question_number,
                        'question': question,
                        'answer': answer,
                        'question_type': question_type,
                        'evaluator_model': evaluator_model,
                        'answer_model': answer_model,
                        'dataset': dataset
                    }
                    
                    # Add scores and reasoning for each metric
                    for category, metrics in evaluation_results.items():
                        # Calculate category average score
                        category_scores = [result['score'] for result in metrics.values()]
                        category_avg_score = sum(category_scores) / len(category_scores) if category_scores else 0
                        
                        # Add category average score
                        category_key = category.replace(' ', '_').replace('-', '_')
                        flattened_result[f'{category_key}_score'] = round(category_avg_score, 2)
                        
                        # Add individual metric scores and reasoning
                        for metric, result in metrics.items():
                            metric_key = metric.replace(' ', '_').replace('-', '_')
                            flattened_result[f'{metric_key}_score'] = result['score']
                            flattened_result[f'{metric_key}_reasoning'] = result['reasoning']
                    
                    all_results.append(flattened_result)
                    
                    # Delay between questions to avoid rate limiting
                    if idx < len(df) - 1:
                        time.sleep(2)
                
                # Save results
                df_results = pd.DataFrame(all_results)
                output_filename = os.path.join(
                    EVALUATION_FOLDER,
                    f"{dataset}_type2_{answer_model}_{evaluator_model}.csv"
                )
                df_results.to_csv(output_filename, index=False)
                
                all_evaluation_results[answer_model][evaluator_model] = df_results
                
                print(f"\n  ✅ Saved: {output_filename}")
                print(f"  ✅ Evaluated {len(all_results)} questions")
                
                # Print summary
                score_cols = [col for col in df_results.columns if col.endswith('_score')]
                print(f"\n  Average Scores:")
                for col in sorted(score_cols):
                    metric_name = col.replace('_score', '').replace('_', ' ').title()
                    avg_score = df_results[col].mean()
                    print(f"    {metric_name:40s}: {avg_score:.2f}/5")
                
            except Exception as e:
                print(f"  ❌ Error evaluating with {evaluator_model}: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        # Delay between answer models
        print(f"\nWaiting 5 seconds before next answer model...")
        time.sleep(5)
    
    print(f"\n{'='*80}")
    print("ALL EVALUATIONS COMPLETE")
    print(f"{'='*80}")
    print(f"Results saved in: {EVALUATION_FOLDER}/")
    print(f"File naming: {dataset}_type2_{{answer_model}}_{{evaluator_model}}.csv")
    print(f"\nGenerated files:")
    for answer_model in answer_models:
        for evaluator_model in evaluator_models:
            filename = f"{dataset}_type2_{answer_model}_{evaluator_model}.csv"
            filepath = os.path.join(EVALUATION_FOLDER, filename)
            if os.path.exists(filepath):
                print(f"  ✅ {filename}")
    
    return all_evaluation_results

# Run the comprehensive evaluation
print("Starting comprehensive evaluation of shopify type2...")
print("This will evaluate 4 answer models × 3 evaluator models = 12 evaluation sets")
print("Each set contains all questions from the answer file\n")

all_results = evaluate_shopify_type2_all_models()
